In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType, DateType, BooleanType, StringType
from pyspark.sql.window import Window

def transform_person(person_df):

    windowSpec_rw = Window.partitionBy("BusinessEntityID").orderBy(F.desc("ModifiedDate"))
    person_df = person_df.withColumn("MiddleName",F.when(F.col("MiddleName").isNull(), F.lit("")).otherwise(F.col("MiddleName"))).withColumn("PersonType ",F.when(~(F.col("PersonType").isin(['ÉM','SP'])), F.lit("OT")).otherwise(F.col("MiddleName"))).withColumn(
		"rw", F.row_number().over(windowSpec_rw))
    person_df = person_df.filter(F.col("rw") == 1).drop("rw")
    person_df = person_df.withColumn("processed_timestamp", F.current_timestamp())

    person_df = person_df.select(    
      F.col("BusinessEntityID").cast(IntegerType()).alias("BusinessEntityID"),
      F.col("AddressID").alias("AddressID"),
      F.col("AddressTypeID").alias("AddressTypeID"),
      F.col("rowguid").alias("rowguid"),
      F.col("ModifiedDate").cast(DateType()).alias("ModifiedDate"),
      F.col("AddressLine1").alias("AddressLine1"),
      F.col("AddressLine2").alias("AddressLine2"),
      F.col("City").alias("City"),
      F.col("StateProvinceID").alias("StateProvinceID"),
      F.col("PostalCode").alias("PostalCode"),
      F.col("SpatialLocation").alias("SpatialLocation"),
      F.col("CountryRegionCode").alias("CountryRegionCode"),
      F.col("Name").alias("Name"),
      F.col("PersonID").alias("PersonID"),
      F.col("ContactTypeID").alias("ContactTypeID"),
      F.col("PasswordHash").alias("PasswordHash"),
      F.col("PasswordSalt").alias("PasswordSalt"),
      F.col("StateProvinceCode").alias("StateProvinceCode"),
      F.col("IsOnlyStateProvinceFlag").cast(BooleanType()).alias("IsOnlyStateProvinceFlag"),
      F.col("TerritoryID").alias("TerritoryID"),
      F.col("PhoneNumberTypeID").alias("PhoneNumberTypeID"),
      F.col("PhoneNumber").alias("PhoneNumber"),
      F.col("EmailAddressID").alias("EmailAddressID"),
      F.col("EmailAddress").alias("EmailAddress"),
      F.col("PersonType").alias("PersonType"),
      F.col("NameStyle").cast(BooleanType()).alias("NameStyle"),
      F.col("Title").alias("Title"),
      F.trim(F.col("FirstName")).alias("FirstName"),
      F.trim(F.col("MiddleName")).alias("MiddleName"),
      F.trim(F.col("LastName")).alias("LastName"),
      F.trim(F.col("Suffix")).alias("Suffix"),
      F.col("EmailPromotion").alias("EmailPromotion"),
      F.col("AdditionalContactInfo").alias("AdditionalContactInfo"),
      F.col("Demographics").alias("Demographics"),
      F.col("_rescued_data").alias("_rescued_data"),
      F.col("processed_timestamp")
    )
                                 
    return person_df




if __name__ == "__main__":

    person_tbl = dbutils.widgets.get("person")
    person_df = df = spark.read.table(person_tbl)
    person_df_tgt = transform_person(person_df)
    person_slv_tbl = dbutils.widgets.get("person_tgt")
    person_df_tgt.write.mode("overwrite").format("delta").partitionBy("ModifiedDate").saveAsTable(person_slv_tbl)
    